### Day 09 · OOP 继承 (L2)

今天进入面向对象三大特性的第一个:**继承**。
你会学会:复用已有类的代码、重写父类行为、理解查找顺序。

> **前置**:Day08 已掌握 class/object/方法/属性
> **关键问题**:当几个类的代码几乎一样,复制粘贴越多改 bug 越痛 —— 能不能**抽共性到上层**,差异留在下层?

#### 为什么需要继承(DRY 原则)

> **痛点**
写几个类发现彼此有大量重复代码。比如 `Book`(书)和 `Magazine`(杂志)都有 `name`/`price`/`info()` 方法,代码几乎完全一样。改一个 bug 要改两处,复制粘贴越多越痛。

> **类比**
法律是父类,地方法规是子类:国家规定"最低工资 A 元",某市可**重写**为更高的 "B 元"。共性留在国家层面,差异留在地方。

> **解释**
继承让子类**自动拥有**父类的属性和方法,在此基础上添加或修改,实现**代码复用**。

> **ASCII 内存图:无继承 vs 有继承**

```
无继承(重复):                    有继承(复用):
┌──────────────┐                ┌──────────────┐
│ Book         │                │ Product(父类) │
│ name, price  │                │ name, price  │
│ info()       │                │ info()       │
└──────────────┘                └──────┬───────┘
┌──────────────┐                      │ 继承
│ Magazine     │                ┌─────┴─────┐
│ name, price  │                │ Book(子类) │
│ info()       │                │ + weight  │
└──────────────┘                └───────────┘
```

#### is-a 关系

> **核心**:继承 = "是一个"(is-a)关系

- `Dog` **is-a** `Animal` → `class Dog(Animal)` ✅
- `Car` **is-a** `Vehicle` → `class Car(Vehicle)` ✅
- `Order` **has-a** `Cart` → 不是继承,是组合(后续 Day11)

> **判断技巧**:把两个类名放进 "X 是一个 Y" 句子,通顺就用继承。

#### 单继承语法 class Child(Parent)

> **解释**
`class 子类(父类):` 括号里写父类名,表示继承关系。Python 支持多继承,本节只讲单继承。

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name
    def breathe(self):
        print(f"{self.name} 在呼吸")

class Dog(Animal):
    def bark(self):
        print(f"{self.name} 汪汪叫")

dog = Dog("旺财")
dog.breathe()
dog.bark()

In [ ]:
# --- 执行过程 ---
# 第 1 行 dog = Dog("旺财"):
#   ① Python 看到 Dog(...),调用 Dog 的构造函数
#   ② Dog 自己没有 __init__,沿"子类→父类"找到 Animal.__init__
#   ③ Animal.__init__(dog, "旺财") → dog.name = "旺财"
#
# 第 2 行 dog.breathe():
#   ① Dog 自己没有 breathe 方法
#   ② 沿"子类→父类"查找,找到 Animal.breathe
#   ③ Animal.breathe(dog) → self = dog → 输出"旺财 在呼吸"
#
# 第 3 行 dog.bark():
#   ① Dog 自己有 bark 方法,直接调用
#   ② Dog.bark(dog) → 输出"旺财 汪汪叫"
dog = Dog("旺财")
dog.breathe()
dog.bark()

> **逐行解剖**
- `class Dog(Animal):` 括号里写父类名
- `dog.breathe()` → Dog 没有 breathe,找到 Animal.breathe → self = dog
- 查找顺序:**子类 → 父类 → 父类的父类**

> **常见错误**
1. **忘写括号**:`class Dog Animal:` → SyntaxError
2. **继承方向反了**:`class Animal(Dog):` → 语义不对

> 问自己:
> - 如果子类和父类都有同名方法,调用时执行哪个?
> - 子类能继承父类的 __init__ 吗?
> - 如果子类想扩展父类属性,该怎么做?

#### super().__init__() 安全调用父类构造函数

> **痛点**
子类想扩展父类属性,又不想重复写父类的绑定逻辑。

> **类比**
儿子继承了父亲的房子(父类属性),再加盖一层(扩展属性),但必须**先**把父亲的房子盖好。

> **解释**
`super()` 返回父类对象,用来**安全地调用父类方法**。

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

class Dog(Animal):
    def __init__(self, name, breed):
        super().__init__(name)
        self.breed = breed
    def bark(self):
        print(f"{self.name}({self.breed}) 汪汪叫")

dog = Dog("旺财", "柯基")
print(dog.name, dog.breed)
dog.bark()

In [ ]:
# --- 执行过程 ---
# 第 1 行 dog = Dog("旺财", "柯基"):
#   ① Python 调用 Dog.__init__(dog, "旺财", "柯基")
#   ② 第 1 行 super().__init__("旺财")
#      → super() 返回 Animal 类
#      → Animal.__init__(dog, "旺财")
#      → dog.name = "旺财"
#   ③ 回到 Dog.__init__,执行 self.breed = "柯基"
#   ④ dog 对象创建完毕,有 name 和 breed 两个属性
dog = Dog("旺财", "柯基")
print(dog.name, dog.breed)
dog.bark()

> **逐行解剖**
- `super().__init__(name)` 等价于 `Animal.__init__(self, name)`,但**更规范**
- 必须先调 `super().__init__()`,再绑定子类自己的属性
- 忘记写 `super().__init__()` 会导致**父类属性未初始化**

> **常见错误**
1. **忘调 super().__init__()**:父类属性丢失 → AttributeError
2. **super() 写错参数**:漏传 name → Animal.__init__ 缺参数

**练习** — 子类扩展属性

定义 `Product` 基类,属性 `name` 和 `price`。
定义子类 `Book(Product)`,新增属性 `author`。
用 `super().__init__()` 继承基类属性。

In [ ]:
# ============ 学员代码区 ============
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price

class Book(Product):
    pass

# b = Book("...", ...)
pass

# ============ 参考答案 ============
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price

class Book(Product):
    def __init__(self, name, price, author):
        super().__init__(name, price)
        self.author = author

b = Book("Python 入门", 59.8, "张三")
print(b.name, b.price, b.author)

#### 方法重写(override)

> **痛点**
父类的方法不能满足子类需要。比如基类 `shipping_cost()` 返回 0,但实体书要运费。

> **解释**
子类重新定义同名方法,调用时**优先执行子类版本**。

In [ ]:
class Product:
    def shipping_cost(self):
        return 0

class PhysicalProduct(Product):
    def __init__(self, weight): self.weight = weight
    def shipping_cost(self):
        return self.weight * 5

class DigitalProduct(Product):
    pass

p = PhysicalProduct(2)
d = DigitalProduct()
print(p.shipping_cost())  # 10
print(d.shipping_cost())  # 0

In [ ]:
# --- 执行过程 ---
# 第 1 行 p.shipping_cost():
#   ① Python 在 PhysicalProduct 中查找 shipping_cost
#   ② 找到了!直接执行 → return self.weight * 5 → 10
#
# 第 2 行 d.shipping_cost():
#   ① DigitalProduct 没有 → 沿继承链找到 Product
#   ② return 0
p = PhysicalProduct(2)
d = DigitalProduct()
print(p.shipping_cost())
print(d.shipping_cost())

**练习** — 电商产品运费体系

定义 `Product` 基类,方法 `shipping_cost()` 返回 0。
子类 `PhysicalProduct` 新增 `weight`,重写 = weight * 8。
子类 `DigitalProduct` 不重写。各创建一个验证。

In [ ]:
# ============ 学员代码区 ============
class Product:
    def shipping_cost(self):
        return 0

class PhysicalProduct(Product):
    pass

# pp = PhysicalProduct(10)
# print(pp.shipping_cost())
pass

# ============ 参考答案 ============
class Product:
    def shipping_cost(self):
        return 0

class PhysicalProduct(Product):
    def __init__(self, weight): self.weight = weight
    def shipping_cost(self):
        return self.weight * 8

class DigitalProduct(Product):
    pass

pp = PhysicalProduct(10)
dp = DigitalProduct()
print(pp.shipping_cost())
print(dp.shipping_cost())

#### super().方法() 保留父类逻辑并扩展

> **场景**
重写方法时,不想完全替换父类,而是**保留 + 扩展**。

In [ ]:
class Employee:
    def __init__(self, name, base):
        self.name, self.base = name, base
    def pay(self):
        return self.base

class Manager(Employee):
    def __init__(self, name, base, bonus):
        super().__init__(name, base)
        self.bonus = bonus
    def pay(self):
        return super().pay() + self.bonus

m = Manager("李四", 8000, 3000)
print(m.pay())

In [ ]:
# --- 执行过程 ---
# 第 1 行 m.pay():
#   ① 调用 Manager.pay(m)
#   ② super().pay() → Employee.pay(m) → return m.base → 8000
#   ③ 回到 Manager.pay,继续 + m.bonus → 8000 + 3000
m = Manager("李四", 8000, 3000)
print(m.pay())

#### MRO 方法解析顺序

> **解释**
多级继承时,查找顺序:**子类 → 父类 → 父类的父类 → object**。
可用 `ClassName.__mro__` 查看。

In [ ]:
class Animal: pass
class Mammal(Animal): pass
class Dog(Mammal): pass

print(Dog.__mro__)

dog = Dog()
print(isinstance(dog, Animal))
print(isinstance(dog, Mammal))

In [ ]:
# --- 执行过程 ---
# 第 1 行 Dog.__mro__:
#   ① 返回 Dog 的方法解析顺序(元组)
#   ② 顺序: Dog → Mammal → Animal → object
#
# 第 2 行 isinstance(dog, Animal):
#   ① 沿继承链向上追溯:Dog → Mammal → Animal ✓
print(Dog.__mro__)
dog = Dog()
print(isinstance(dog, Animal))

> **常见错误**
1. **MRO 混乱**:多重继承时顺序不清 → 用 `__mro__` 查看
2. **isinstance 过度使用**:用它做类型分发是反模式(应使用多态)

#### isinstance 反模式(为什么不该用)

> **反模式代码**(能跑但不好):
```
def pay(emp):
    if isinstance(emp, Manager):
        return emp.base + emp.bonus
    elif isinstance(emp, Sales):
        return emp.base + emp.commission
```

> **3 个致命问题**:
1. 每新增一种员工类型,都要改 `pay` 函数
2. `pay` 函数越来越长,难维护
3. 违反"开闭原则"

> **正确做法**:用多态(留到 Day10)

#### 综合练习:员工薪资系统

1. 基类 `Employee`:属性 `name`、`base_salary`,方法 `pay()` 返回 base_salary
2. 子类 `Sales`:新增 `commission`,**重写** pay()
3. 子类 `Manager`:新增 `bonus`,**重写** pay()
4. 用 `super().__init__()` 调用父类构造
5. 写函数 `show_pay(emp)` 传入任意员工,打印 pay()

In [ ]:
# ============ 学员代码区 ============
class Employee:
    def __init__(self, name, base_salary):
        self.name = name
        self.base_salary = base_salary
    def pay(self):
        return self.base_salary

# class Sales(Employee): ...
# class Manager(Employee): ...
# def show_pay(emp): ...
pass

# ============ 参考答案 ============
class Employee:
    def __init__(self, name, base_salary):
        self.name = name
        self.base_salary = base_salary
    def pay(self):
        return self.base_salary

class Sales(Employee):
    def __init__(self, name, base, commission):
        super().__init__(name, base)
        self.commission = commission
    def pay(self):
        return super().pay() + self.commission

class Manager(Employee):
    def __init__(self, name, base, bonus):
        super().__init__(name, base)
        self.bonus = bonus
    def pay(self):
        return super().pay() + self.bonus

def show_pay(emp):
    if isinstance(emp, Employee):
        print(f"{emp.name} 薪资: {emp.pay()}")

for e in [Employee("张三", 5000), Sales("李四", 5000, 2000)]:
    show_pay(e)

**今日小结**

| 概念 | 解决的痛点 |
| --- | --- |
| 继承 | 复用已有类代码,避免重复 |
| super() | 安全调用父类构造函数 |
| 方法重写 | 子类用自定义行为替换父类 |
| MRO | 多级继承时查找顺序清晰 |
| isinstance 反模式 | 用 if-elif 分发每加类型都要改 |

**更多练习**
- 当堂练:course/lesson09/in_class/practice01-06.py
- 课后作业:course/lesson09/homework/task01-03.py

**预告**:Day10 将重构 `show_pay`,用多态去掉 isinstance!